## **Dependencies**

In [1]:
!pip install -q -U rank-bm25 nltk tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [2]:
#!pip install -q -U transformers accelerate torch

## **Imports**

In [2]:
import json
import os
import pickle
import re
from typing import Dict, List, Set, Any

import math
import numpy as np
from tqdm import tqdm
from collections import Counter

import torch
from sentence_transformers import CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSequenceClassification
from rank_bm25 import BM25Okapi

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

In [3]:
nltk.pathsec.ALLOW_PROXIED_FETCH = True
nltk.download('stopwords', quiet=True)

True

In [4]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("my_hf_token")
login(token=hf_token)

## **Configuration Paths**

In [5]:
DATA_DIR = "/kaggle/input/datasets/anarvaaa/original-scifact-data"
DEV_CLAIMS_PATH = os.path.join(DATA_DIR, "claims_dev.jsonl")
INDEX_PKL_PATH = os.path.join(DATA_DIR, "scifact_bm25_index.pkl")
SCIBERT_MODEL_PATH = "/kaggle/input/datasets/anarvaaa/scibert-verdict-model-files"
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {DEVICE}")
EXPANSION_MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-base"
CANDIDATE_POOL_SIZE = 50 
FINAL_TOP_K = 5

Using compute device: cuda


## **Preprocessing**

In [7]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def tokenize(text: str, remove_stopwords: bool = True, use_stemming: bool = True) -> List[str]:
    # Clean non-alphanumeric noise to protect BM25 token matches
    tokens = re.findall(r"\b[a-zA-Z0-9]+(?:-[a-zA-Z0-9]+)*\b", text.lower())
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]
    if use_stemming:
        tokens = [stemmer.stem(t) for t in tokens]
    return tokens

## **Load Index**

In [8]:
print(f"Loading BM25 index from: {INDEX_PKL_PATH}")
with open(INDEX_PKL_PATH, "rb") as f:
    bm25_artifacts = pickle.load(f)

bm25: BM25Okapi = bm25_artifacts["bm25_model"]
doc_ids: List[int] = bm25_artifacts["doc_ids"]
doc_metadata: Dict[int, Dict[str, Any]] = bm25_artifacts["doc_metadata"]

print(f"Successfully loaded index with {len(doc_ids):,} indexed documents.")

Loading BM25 index from: /kaggle/input/datasets/anarvaaa/original-scifact-data/scifact_bm25_index.pkl
Successfully loaded index with 5,183 indexed documents.


## **Load Models**

In [9]:
print("Loading trained SciBERT for Verdict")

scibert_tokenizer = AutoTokenizer.from_pretrained(
    SCIBERT_MODEL_PATH
)

scibert_model = AutoModelForSequenceClassification.from_pretrained(
    SCIBERT_MODEL_PATH
)

scibert_model.to(DEVICE)
scibert_model.eval()

print("SciBERT loaded successfully.")
print(scibert_model.config.id2label)

Loading trained SciBERT for Verdict


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

SciBERT loaded successfully.
{0: 'CONTRADICT', 1: 'SUPPORT'}


In [10]:
print(f"Loading Cross-Encoder Reranker: {RERANKER_MODEL_NAME} for Re-ranking")
reranker = CrossEncoder(RERANKER_MODEL_NAME, max_length=512, device=DEVICE)

Loading Cross-Encoder Reranker: BAAI/bge-reranker-base for Re-ranking


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=hf_token,
    dtype=torch.float16,      # NOT torch_dtype (deprecated, as you already found)
    device_map="auto",
)
model.eval()

DEVICE = next(model.parameters()).device
NUM_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
print(f"Loaded {MODEL_NAME}: {NUM_LAYERS} layers, hidden_dim={HIDDEN_DIM}, device={DEVICE}")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Loaded meta-llama/Llama-3.2-3B-Instruct: 28 layers, hidden_dim=3072, device=cuda:0


In [ ]:
'''
print(f"Loading {LLAMA_MODEL_NAME} for inference")

llama_tokenizer = AutoTokenizer.from_pretrained(
    LLAMA_MODEL_NAME
)

llama_model = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

llama_model.eval()

print("Llama loaded.")
'''

## **Helper Functions**

In [12]:
def retrieve_bm25(query_str: str, k: int = CANDIDATE_POOL_SIZE) -> List[Dict[str, Any]]:
    tokens = tokenize(query_str)
    scores = bm25.get_scores(tokens)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    
    results = []
    for rank, idx in enumerate(top_indices, start=1):
        d_id = doc_ids[idx]
        results.append({
            "doc_id": d_id,
            "rank": rank,
            "score": round(float(scores[idx]), 4),
            "title": doc_metadata[d_id]["title"],
            "abstract_text": doc_metadata[d_id]["abstract_text"]
        })
    return results

In [13]:
def rerank_documents_hybrid(
    expanded_query: str, 
    candidate_docs: List[Dict[str, Any]], 
    alpha: float = 0.7
) -> List[Dict[str, Any]]:
    """
    Hybrid Re-ranking: 
        Combines Min-Max normalized Cross-Encoder scores with BM25 scores to prevent query drift.
    """
    if not candidate_docs:
        return []

    # 1. Match against Expanded Query
    pairs = [
        [expanded_query, f"{doc['title']} {doc['abstract_text']}".strip()] 
        for doc in candidate_docs
    ]
    
    ce_scores = reranker.predict(pairs)

    # 2. Normalize BM25 and Cross-Encoder scores for linear interpolation
    bm25_raw = [d["score"] for d in candidate_docs]
    ce_raw = list(ce_scores)

    min_bm25, max_bm25 = min(bm25_raw), max(bm25_raw)
    min_ce, max_ce = min(ce_raw), max(ce_raw)

    bm25_norm = [(s - min_bm25) / (max_bm25 - min_bm25 + 1e-6) for s in bm25_raw]
    ce_norm = [(s - min_ce) / (max_ce - min_ce + 1e-6) for s in ce_raw]

    # 3. Score Fusion: Final_Score = alpha * CE_norm + (1 - alpha) * BM25_norm
    reranked_docs = []
    for idx, doc in enumerate(candidate_docs):
        doc_copy = doc.copy()
        fusion_score = (alpha * ce_norm[idx]) + ((1 - alpha) * bm25_norm[idx])
        doc_copy["ce_score"] = float(ce_scores[idx])
        doc_copy["fusion_score"] = float(fusion_score)
        reranked_docs.append(doc_copy)

    return sorted(reranked_docs, key=lambda x: x["fusion_score"], reverse=True)

In [14]:
def get_first_400_tokens(text):
    tokens = scibert_tokenizer.tokenize(text)

    tokens = tokens[:400]

    return scibert_tokenizer.convert_tokens_to_string(tokens)

In [15]:
def predict_verdict(claim, document_text):

    document_prefix = get_first_400_tokens(
        document_text
    )

    inputs = scibert_tokenizer(
        claim,
        document_prefix,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = scibert_model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    predicted_id = torch.argmax(
        probabilities
    ).item()

    label = scibert_model.config.id2label[
        predicted_id
    ]

    return {
        "verdict": label,
        "contradict_probability": float(probabilities[0]),
        "support_probability": float(probabilities[1])
    }

In [16]:
def build_summary_prompt(query, top_5_documents):

    evidence_blocks = []

    for doc in top_5_documents:

        block = f"""
DOCUMENT {doc['rank']}
Document ID: {doc['doc_id']}
Title: {doc['title']}

SciBERT verdict: {doc['verdict']}
Support probability: {doc['support_probability']:.4f}
Contradict probability: {doc['contradict_probability']:.4f}

ABSTRACT:
{doc['abstract_text']}
"""

        evidence_blocks.append(block)

    evidence = "\n".join(evidence_blocks)

    prompt = f"""
You are a scientific evidence summarization assistant.

Your task is to answer the CLAIM using ONLY the information
contained in the five provided document abstracts.

CLAIM:
{query}

The documents have been ranked by a retrieval and reranking
system. A SciBERT classifier has also predicted whether each
document SUPPORTS or CONTRADICTS the claim.

Use these verdicts as guidance, but read the abstracts yourself.

{evidence}

Instructions:

1. Determine which documents support the claim and which
   contradict it.
2. Synthesize the evidence across the documents.
3. Produce a concise, scientifically grounded summary.
4. Clearly indicate whether the overall evidence supports,
   contradicts, or is mixed with respect to the claim.
5. Mention the relevant document numbers when describing
   evidence, e.g. [Document 1].
6. Do NOT introduce facts that are not present in the
   provided abstracts.
7. If the abstracts do not provide enough evidence to reach
   a conclusion, explicitly say so.
8. Do not treat the SciBERT verdict as evidence itself; the
   abstracts are the evidence.

Return only the grounded summary.
"""

    return prompt

In [17]:
def generate_grounded_summary(query, top_5_documents):

    prompt = build_summary_prompt(
        query,
        top_5_documents
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a scientific evidence summarization "
                "assistant. Be strictly grounded in the "
                "provided documents."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=16000
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=False,
            temperature=None,
            top_p=None
        )

    generated_tokens = outputs[
        0
    ][inputs["input_ids"].shape[-1]:]

    summary = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return summary.strip()

In [18]:
'''

def retrieve_and_verdict(query):

    # --------------------------------------------------
    # 1. BM25 retrieval
    # --------------------------------------------------
    bm25_docs = retrieve_bm25(
        query,
        k=CANDIDATE_POOL_SIZE
    )

    # --------------------------------------------------
    # 2. Cross-encoder reranking
    # --------------------------------------------------
    reranked_docs = rerank_documents_hybrid(
        query,
        bm25_docs,
        alpha=0.7
    )

    # --------------------------------------------------
    # 3. Take final top 5
    # --------------------------------------------------
    top_5_docs = reranked_docs[:FINAL_TOP_K]

    # --------------------------------------------------
    # 4. SciBERT verdict
    # --------------------------------------------------
    final_results = []

    for rank, doc in enumerate(top_5_docs, start=1):

        verdict_result = predict_verdict(
            query,
            doc["abstract_text"]
        )

        final_results.append({
            "rank": rank,
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "abstract_text": doc["abstract_text"],
            "bm25_score": doc["score"],
            "ce_score": doc["ce_score"],
            "fusion_score": doc["fusion_score"],
            "verdict": verdict_result["verdict"],
            "contradict_probability": verdict_result[
                "contradict_probability"
            ],
            "support_probability": verdict_result[
                "support_probability"
            ]
        })

    summary = generate_grounded_summary(
        query,
        final_results
    )

    return {
        "query": query,
        "top_5_documents": final_results
    }

'''

'\n\ndef retrieve_and_verdict(query):\n\n    # --------------------------------------------------\n    # 1. BM25 retrieval\n    # --------------------------------------------------\n    bm25_docs = retrieve_bm25(\n        query,\n        k=CANDIDATE_POOL_SIZE\n    )\n\n    # --------------------------------------------------\n    # 2. Cross-encoder reranking\n    # --------------------------------------------------\n    reranked_docs = rerank_documents_hybrid(\n        query,\n        bm25_docs,\n        alpha=0.7\n    )\n\n    # --------------------------------------------------\n    # 3. Take final top 5\n    # --------------------------------------------------\n    top_5_docs = reranked_docs[:FINAL_TOP_K]\n\n    # --------------------------------------------------\n    # 4. SciBERT verdict\n    # --------------------------------------------------\n    final_results = []\n\n    for rank, doc in enumerate(top_5_docs, start=1):\n\n        verdict_result = predict_verdict(\n            

In [19]:
def retrieve_rerank_verdict_summarize(query):

    # ----------------------------------------
    # 1. BM25
    # ----------------------------------------
    bm25_docs = retrieve_bm25(
        query,
        k=CANDIDATE_POOL_SIZE
    )

    # ----------------------------------------
    # 2. Cross-encoder reranking
    # ----------------------------------------
    reranked_docs = rerank_documents_hybrid(
        query,
        bm25_docs,
        alpha=0.7
    )

    # ----------------------------------------
    # 3. Top 5
    # ----------------------------------------
    top_5_docs = reranked_docs[:FINAL_TOP_K]

    # ----------------------------------------
    # 4. SciBERT verdict
    # ----------------------------------------
    final_results = []

    for rank, doc in enumerate(top_5_docs, start=1):

        verdict_result = predict_verdict(
            query,
            doc["abstract_text"]
        )

        final_results.append({
            "rank": rank,
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "abstract_text": doc["abstract_text"],

            "bm25_score": doc["score"],
            "ce_score": doc["ce_score"],
            "fusion_score": doc["fusion_score"],

            "verdict": verdict_result["verdict"],
            "contradict_probability":
                verdict_result["contradict_probability"],
            "support_probability":
                verdict_result["support_probability"]
        })

    # ----------------------------------------
    # 5. Llama grounded summary
    # ----------------------------------------
    summary = generate_grounded_summary(
        query,
        final_results
    )

    return {
        "query": query,
        "top_5_documents": final_results,
        "grounded_summary": summary
    }

## **Test Run**

In [20]:
query = (
    "A deficiency of vitamin B12 increases "
    "blood levels of homocysteine."
)

result = retrieve_rerank_verdict_summarize(
    query
)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [21]:
print("=" * 100)
print("QUERY")
print("=" * 100)

print(result["query"])

print("\n" + "=" * 100)
print("TOP 5 DOCUMENTS + SCIBERT VERDICTS")
print("=" * 100)

for doc in result["top_5_documents"]:

    print(f"\nDocument {doc['rank']}")
    print(f"ID: {doc['doc_id']}")
    print(f"Title: {doc['title']}")
    print(f"Verdict: {doc['verdict']}")
    print(
        f"Support probability: "
        f"{doc['support_probability']:.4f}"
    )
    print(
        f"Contradict probability: "
        f"{doc['contradict_probability']:.4f}"
    )

print("\n" + "=" * 100)
print("GROUNDED SUMMARY")
print("=" * 100)

print(result["grounded_summary"])

QUERY
A deficiency of vitamin B12 increases blood levels of homocysteine.

TOP 5 DOCUMENTS + SCIBERT VERDICTS

Document 1
ID: 18557974
Title: British Journal of Nutrition (2003), 89, 295–301 q The Authors 2003 DOI: 10.1079/BJN2002776 Plasma homocysteine concentration is decreased by dietary intervention*
Verdict: SUPPORT
Support probability: 0.5097
Contradict probability: 0.4903

Document 2
ID: 18256197
Title: Homocysteine and the risk of ischemic stroke in a triethnic cohort: the NOrthern MAnhattan Study.
Verdict: CONTRADICT
Support probability: 0.4228
Contradict probability: 0.5772

Document 3
ID: 33409100
Title: Effect of homocysteine lowering on mortality and vascular disease in advanced chronic kidney disease and end-stage renal disease: a randomized controlled trial.
Verdict: CONTRADICT
Support probability: 0.3810
Contradict probability: 0.6190

Document 4
ID: 42441846
Title: Gene--nutrition interactions in coronary artery disease: correlation between the MTHFR C677T polymorphism